# Bài thực hành 1: Xây dựng giao diện người dùng cơ bản cho agent

Trong bài học này, bạn sẽ kết nối một agent LangChain với giao diện chat React bằng cách sử dụng CopilotKit và giao thức AG-UI - đây cũng chính là mô hình tiêu chuẩn được sử dụng để kết nối bất kỳ backend agent nào với bất kỳ frontend nào.


## Mục tiêu học tập

1. **Chạy một agent LangChain** - Khởi chạy một backend tương thích với AG-UI bằng FastAPI.
2. **Thiết lập CopilotKit** - Kết nối frontend React với agent của bạn thông qua CopilotRuntime.
3. **Hoán đổi backend của agent** - Chuyển đổi qua lại giữa LangChain/OpenAI và Google ADK/Gemini mà không cần thay đổi mã nguồn UI.

## Trước khi bạn bắt đầu

Bài học này sử dụng lệnh `%%writefile` để lưu code trực tiếp vào dự án frontend trên ổ đĩa. Ứng dụng đang chạy sẽ tự động nhận diện các thay đổi đó - vì vậy bạn sẽ thấy giao diện UI của mình cập nhật theo thời gian thực khi bạn chạy các khối lệnh dưới đây.

<div style="background-color:#fff1d7; padding:15px; border-left: 4px solid #f0b429; border-radius: 6px;">
<b>Bạn đã hoàn thành bài học này rồi?</b> Nếu bạn muốn bắt đầu lại notebook này từ đầu, hãy chạy khối lệnh bên dưới để khôi phục tất cả các file về trạng thái ban đầu. <b>Hãy bỏ qua bước này nếu đây là lần đầu tiên bạn học bài này.</b>
</div>

In [ ]:
# from helper import reset_lesson
# reset_lesson(2)

## Bạn sẽ xây dựng những gì

Một giao diện chat được hỗ trợ bởi agent LangChain, kết nối thông qua CopilotKit. Đến cuối bài học, bạn sẽ có thể trò chuyện trực tiếp với agent của mình trên trình duyệt:

<img src="images/copilotkit-chat.png" alt="CopilotKit Chat" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

## Cài đặt các thư viện phụ thuộc

Hãy chạy hai khối lệnh tiếp theo để xác nhận mọi thứ đã sẵn sàng.

In [1]:
# ẩn các tin nhắn cảnh báo
import warnings
warnings.filterwarnings("ignore")

In [2]:
from helper import install_frontend
install_frontend()

Installing frontend dependencies ...
npm WARN EBADENGINE Unsupported engine {
npm WARN EBADENGINE   package: '@langchain/core@1.2.3',
npm WARN EBADENGINE   required: { node: '>=20' },
npm WARN EBADENGINE   current: { node: 'v18.19.1', npm: '9.2.0' }
npm WARN EBADENGINE }
npm WARN EBADENGINE Unsupported engine {
npm WARN EBADENGINE   package: 'langchain@1.5.3',
npm WARN EBADENGINE   required: { node: '>=20' },
npm WARN EBADENGINE   current: { node: 'v18.19.1', npm: '9.2.0' }
npm WARN EBADENGINE }
npm WARN EBADENGINE Unsupported engine {
npm WARN EBADENGINE   package: '@tailwindcss/oxide@4.3.2',
npm WARN EBADENGINE   required: { node: '>= 20' },
npm WARN EBADENGINE   current: { node: 'v18.19.1', npm: '9.2.0' }
npm WARN EBADENGINE }
npm WARN EBADENGINE Unsupported engine {
npm WARN EBADENGINE   package: '@vercel/oidc@3.2.0',
npm WARN EBADENGINE   required: { node: '>= 20' },
npm WARN EBADENGINE   current: { node: 'v18.19.1', npm: '9.2.0' }
npm WARN EBADENGINE }
npm WARN EBADENGINE Unsuppo

Bây giờ bạn sẽ tải các API key cho các mô hình mà agent của bạn sẽ sử dụng.

In [2]:
from helper import load_api_keys
load_api_keys()

✓ OpenAI API key loaded
✓ Google API key loaded


Bạn sẽ cần tự lấy các khóa API của riêng mình:

- **OpenAI** (dành cho phần chính của bài học): [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
- **Google AI** (dành cho phần nâng cao): [aistudio.google.com/apikey](https://aistudio.google.com/app/apikey)

## Xây dựng agent

### Khởi động máy chủ

CopilotKit kết nối với agent của bạn thông qua một endpoint HTTP tương thích với AG-UI. Tại đây, bạn sẽ khởi động một máy chủ FastAPI và gắn `LangGraphAGUIAgent` vào đó.

In [3]:
from fastapi import FastAPI

# Các thư viện phụ thuộc của CopilotKit và AG-UI dành cho máy chủ AG-UI
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import LangGraphAGUIAgent
from langchain.agents import create_agent

# Hàm hỗ trợ đơn giản để khởi động máy chủ và quản lý xung đột cổng
from helper import start_server

# Tích hợp endpoint AG-UI vào ứng dụng FastAPI
app = FastAPI()
graph = create_agent("openai:gpt-4.1")
agent = LangGraphAGUIAgent(
    name="lesson2_agent",
    description="Lesson 2 chart agent",
    graph=graph
)
add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")

# Khởi động máy chủ
start_server(app, port=8002)

✓ Server running at http://localhost:8002


AG-UI là gì? [AG-UI](https://docs.ag-ui.com) là một giao thức mở dùng để kết nối các backend agent với frontend. CopilotKit sử dụng giao thức này ở chế độ nền. Bạn sẽ tìm hiểu chi tiết hơn về cách nó hoạt động ở cuối bài học này.

### Định nghĩa agent

Tại đây, bạn sẽ tạo một agent LangChain sử dụng mô hình OpenAI, memory checkpointer và middleware của CopilotKit.

Hãy chạy lại khối lệnh này bất cứ khi nào bạn muốn thay đổi cấu hình của agent - dòng lệnh `agent.graph = ...` sẽ tải lại đồ thị mà không cần phải khởi động lại máy chủ.

In [4]:
from copilotkit import CopilotKitMiddleware

# Import các thư viện agent của LangChain
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

# Tạo một agent LangChain dựng sẵn
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[],
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt="Bạn là một trợ lý hữu ích"
)

# Cập nhật đồ thị của agent
agent.graph = graph

`CopilotKitMiddleware()` chính là thứ kết nối agent của bạn với CopilotKit - nó cho phép mô hình phát hiện và gọi các công cụ ở frontend, những thứ mà bạn sẽ xây dựng ở bài thực hành 2.

Nếu không có middleware này, agent sẽ chỉ nhìn thấy các công cụ được định nghĩa ở phía backend.

## Bắt đầu với CopilotKit
CopilotKit cung cấp cả UI hoàn toàn không giao diện và các thành phần React dựng sẵn dành cho giao diện của agent. Trong bài học này, bạn sẽ sử dụng ba phần:

- **`CopilotRuntime`**: Một cầu nối an toàn giúp kết nối frontend của bạn với bất kỳ backend agent nào.
- **`CopilotKit`**: Provider của React để cấu hình kết nối runtime cho ứng dụng của bạn.
- **`CopilotChat`**: Một giao diện chat đầy đủ tính năng, có thể tùy chỉnh dành cho các agent của bạn.

### Khởi động frontend

Bạn sẽ sử dụng một ứng dụng được dựng sẵn làm nền tảng cho các bài học. Khi bạn thực hiện thay đổi, ứng dụng sẽ tự động ghi nhận các thay đổi đó và hiển thị cho bạn.

Hãy chạy hai khối lệnh tiếp theo để khởi động máy chủ lập trình và mở bản xem trước trực tiếp.

In [5]:
from helper import start_frontend
start_frontend(port=3002)

Starting frontend on port 3002 ...
✓ App running at http://localhost:3002

Read the logs: /home/cuong-ta/Documents/build-interactive-agents-with-generative-ui/1-building-a-basic-agent-ui/frontend/dev-logs.txt


Cuối cùng, bạn có thể mở bản xem trước trực tiếp của ứng dụng đang chạy.

In [ ]:
from helper import display_app
display_app(port=3002)

Tiếp theo, bạn sẽ thiết lập một khung chat để trò chuyện với agent trong ứng dụng này. Để bắt đầu, đây sẽ chỉ là một ứng dụng chat trống.

### Thiết lập `CopilotRuntime`

`CopilotRuntime` là cầu nối an toàn giữa frontend và backend agent của bạn. Tại đây, bạn sẽ đăng ký agent LangChain của mình làm agent `default`:

In [7]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import { LangGraphHttpAgent } from "@ag-ui/langgraph";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";

const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

Overwriting frontend/server.ts


Tại sao lại dùng `v2`? Các gói của CopilotKit xuất cả API v1 và v2. Đường dẫn import `v2` cung cấp cho bạn các hook và component mới nhất. Tất cả mã nguồn trong khóa học này đều sử dụng v2.

### Bao bọc ứng dụng của bạn trong Provider `CopilotKit`

Provider `CopilotKit` kết nối ứng dụng React của bạn với runtime thông qua `runtimeUrl`. Hãy đặt nó trong `main.tsx` để bao bọc toàn bộ ứng dụng - hoặc bao bọc xung quanh các component cụ thể nếu bạn muốn. 

In [8]:
%%writefile frontend/src/main.tsx

import { StrictMode } from "react";
import { createRoot } from "react-dom/client";
import { CopilotKit } from "@copilotkit/react-core/v2";
import "@copilotkit/react-core/v2/styles.css";
import "./globals.css";
import App from "./App";

createRoot(document.getElementById("root")!).render(
  <StrictMode>
    <main className="h-screen w-screen">
      <CopilotKit runtimeUrl="/api/copilotkit" useSingleEndpoint={false}>
        <App />
      </CopilotKit>
    </main>
  </StrictMode>,
);

Overwriting frontend/src/main.tsx


### Thiết lập component `CopilotChat`

Cuối cùng, hãy thêm component `CopilotChat` và chỉ định nó kết nối với agent `default` mà bạn đã đăng ký trong `CopilotRuntime`.

In [9]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";

const agentId = "default";

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

Overwriting frontend/src/App.tsx


Lưu ý: Bạn cũng có thể sử dụng CopilotKit ở chế độ headless (không cần giao diện chat dựng sẵn) thông qua hook `useAgent`. Trong bài học này, bạn sẽ sử dụng component `CopilotChat` có sẵn để đơn giản hóa.

### Trải nghiệm thực tế!

Ứng dụng của bạn hiện đã có giao diện chat hoạt động. Chạy khối lệnh bên dưới để mở bản xem trước và thử trò chuyện với agent của bạn.

In [ ]:
from helper import display_app
display_app(port=3002)

## Phần nâng cao - Kết nối với Google ADK

Trong phần này, bạn sẽ thêm một backend agent thứ hai sử dụng Google ADK và Gemini - và chuyển sang sử dụng nó mà không cần thay đổi bất kỳ dòng mã UI nào.

Chú ý: Phần này yêu cầu phải có `GEMINI_API_KEY`. Bạn có thể lấy API key Gemini của riêng mình từ [Google AI Studio](https://aistudio.google.com/app/apikey).

### Thiết lập một agent ADK

Trước tiên, hãy tạo và khởi động một agent Google ADK trên cổng `8009`:

In [12]:
from fastapi import FastAPI
from ag_ui_adk import ADKAgent, add_adk_fastapi_endpoint
from google.adk.agents import LlmAgent
from helper import start_server

gemini_agent = LlmAgent(
    name="assistant",
    model="gemini-2.5-flash",
    instruction="Hãy tỏ ra hữu ích và vui vẻ!"
)

adk_agent = ADKAgent(
    adk_agent=gemini_agent,
    app_name="demo_app",
    user_id="demo_user",
    session_timeout_seconds=3600,
    use_in_memory_services=True
)

app_adk = FastAPI()
add_adk_fastapi_endpoint(app_adk, adk_agent, path="/")

start_server(app_adk, port=8009)

✓ Server running at http://localhost:8009


### Cập nhật `CopilotRuntime` để đăng ký agent

Bây giờ, hãy cập nhật runtime để đăng ký agent Gemini song song với agent mặc định:

In [13]:
%%writefile frontend/server.ts

import { CopilotRuntime, createCopilotEndpoint } from "@copilotkit/runtime/v2";
import { HttpAgent } from "@ag-ui/client";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";
import { serve } from "@hono/node-server";

const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

const adkAgent = new HttpAgent({
  url: process.env.ADK_AGENT_URL || "http://localhost:8009",
});

const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
    gemini: adkAgent,
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

Overwriting frontend/server.ts


### Cập nhật frontend để sử dụng Gemini

Để chuyển đổi mô hình, hãy cập nhật `agentId` trong `frontend/src/App.tsx`:
- `"default"` -> LangChain/OpenAI
- `"gemini"` -> ADK/Gemini

In [14]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";

export const agentId = "gemini";

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

Overwriting frontend/src/App.tsx


### Hiển thị ứng dụng

Khung chat của bạn hiện đã được kết nối với backend Gemini. Hãy thử hỏi nó một vài câu hỏi và chú ý những sự khác biệt trong cách nó hành xử.

In [15]:
from helper import display_app
display_app(port=3002)

## AG-UI là gì?

AG-UI (Agent-User Interaction) là một giao thức mở, dựa trên sự kiện (event-based) để kết nối các backend agent với frontend. Nó chuẩn hóa luồng tin nhắn chat, các lệnh gọi công cụ, cập nhật trạng thái và stream các token qua HTTP.

Bạn vừa thấy điều này được áp dụng thực tế - dưới đây là lý do tại sao nó quan trọng:
- CopilotKit có thể giao tiếp với bất kỳ backend nào triển khai AG-UI.
- Bạn đã chuyển đổi từ LangChain/OpenAI sang ADK/Gemini chỉ bằng một thay đổi cấu hình - không cần viết lại giao diện UI.
- Hành vi stream và sử dụng công cụ vẫn nhất quán trên mọi framework.


<img src="images/protocols.png" alt="AG-UI protocol diagram" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />


## Những gì bạn đã học được

- Cách chạy một agent LangChain phía sau một endpoint AG-UI và kết nối nó với CopilotKit.
- Cách tải lại đồ thị agent trong quá trình phát triển.
- Cách sử dụng cùng một frontend để chuyển đổi qua lại giữa các backend LangChain/OpenAI và ADK/Gemini.

## Bước tiếp theo

Trong **bài thực hành 2**, bạn sẽ tập trung vào **controlled GenUI**:
- Đăng ký các công cụ frontend định kiểu với `useComponent()`
- Render đầu ra của các công cụ có cấu trúc trong chat
- Giữ cho hành vi UI do mô hình điều khiển luôn có thể dự đoán được và an toàn